In [1]:
year           = 2024
bronze_db      = "bronze"   
silver_table   = "taxi_trips"
write_mode     = "overwrite" 

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, StringType, TimestampType, ShortType
)

spark = SparkSession.builder.getOrCreate()

BRONZE_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/bronze.Lakehouse"
)

BRONZE_TAXI_PATH    = f"{BRONZE_BASE}/Files/taxi/"

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 4, Finished, Available, Finished, False)

In [3]:
raw_df = (
    spark.read
    .parquet(BRONZE_TAXI_PATH + f"yellow_tripdata_{year}-*.parquet")
)

raw_count = raw_df.count()
print(f"Bronze rows loaded : {raw_count:,}")
print(f"Columns            : {raw_df.columns}")

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 5, Finished, Available, Finished, False)

Bronze rows loaded : 41,169,720
Columns            : ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


In [4]:
typed_df = (
    raw_df
    .withColumn("pickup_datetime",      F.col("tpep_pickup_datetime").cast(TimestampType()))
    .withColumn("dropoff_datetime",     F.col("tpep_dropoff_datetime").cast(TimestampType()))
    .withColumn("passenger_count",      F.col("passenger_count").cast(ShortType()))
    .withColumn("trip_distance",        F.col("trip_distance").cast(DoubleType()))
    .withColumn("rate_code_id",         F.col("RatecodeID").cast(ShortType()))
    .withColumn("store_and_fwd_flag",   F.trim(F.col("store_and_fwd_flag").cast(StringType())))
    .withColumn("pu_location_id",       F.col("PULocationID").cast(IntegerType()))
    .withColumn("do_location_id",       F.col("DOLocationID").cast(IntegerType()))
    .withColumn("payment_type",         F.col("payment_type").cast(ShortType()))
    .withColumn("fare_amount",          F.col("fare_amount").cast(DoubleType()))
    .withColumn("extra",                F.col("extra").cast(DoubleType()))
    .withColumn("mta_tax",              F.col("mta_tax").cast(DoubleType()))
    .withColumn("tip_amount",           F.col("tip_amount").cast(DoubleType()))
    .withColumn("tolls_amount",         F.col("tolls_amount").cast(DoubleType()))
    .withColumn("improvement_surcharge",F.col("improvement_surcharge").cast(DoubleType()))
    .withColumn("total_amount",         F.col("total_amount").cast(DoubleType()))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast(DoubleType()))
    .withColumn("airport_fee",          F.col("airport_fee").cast(DoubleType()))
    .select(
        "pickup_datetime", "dropoff_datetime", "passenger_count",
        "trip_distance", "rate_code_id", "store_and_fwd_flag",
        "pu_location_id", "do_location_id", "payment_type",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee",
    )
)

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 6, Finished, Available, Finished, False)

In [5]:
dedup_df    = typed_df.dropDuplicates()
dupes_removed = raw_count - dedup_df.count()
print(f"Duplicate rows removed : {dupes_removed:,}")

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 7, Finished, Available, Finished, False)

Duplicate rows removed : 4


In [6]:
year_start = F.lit(f"{year}-01-01 00:00:00").cast(TimestampType())
year_end   = F.lit(f"{year}-12-31 23:59:59").cast(TimestampType())

clean_df = (
    dedup_df
    .filter(F.col("fare_amount")    >  0)
    .filter(F.col("trip_distance")  >  0)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("pickup_datetime").between(year_start, year_end))
    .filter(F.col("dropoff_datetime") > F.col("pickup_datetime"))
    .filter(
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"))
        .between(60, 43200)  # 1 min → 12 hours
    )
)

filtered_out = dedup_df.count() - clean_df.count()
print(f"Rows removed by business rules : {filtered_out:,}")

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 8, Finished, Available, Finished, False)

Rows removed by business rules : 5,678,773


In [7]:
enriched_df = (
    clean_df
    .withColumn("pickup_date",
        F.to_date("pickup_datetime"))
    .withColumn("pickup_hour",
        F.hour("pickup_datetime").cast(ShortType()))
    .withColumn("day_of_week",
        F.dayofweek("pickup_datetime").cast(ShortType()))   # 1=Sun … 7=Sat
    .withColumn("trip_duration_minutes",
        ((F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60)
        .cast(DoubleType()))
    .withColumn("_year",
        F.lit(year).cast(ShortType()))
)

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 9, Finished, Available, Finished, False)

In [8]:
silver_count = enriched_df.count()
print(f"\nRows written to silver : {silver_count:,}")

(
    enriched_df.write
    .format("delta")
    .mode(write_mode)
    .partitionBy("pickup_date")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"[OK] silver.{silver_table} written  (mode={write_mode})")

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 10, Finished, Available, Finished, False)


Rows written to silver : 35,490,943
[OK] silver.taxi_trips written  (mode=overwrite)


In [9]:
spark.sql(f"""
    SELECT
        MIN(pickup_date)                        AS earliest_date,
        MAX(pickup_date)                        AS latest_date,
        COUNT(*)                                AS total_trips,
        ROUND(AVG(fare_amount),   2)            AS avg_fare_usd,
        ROUND(AVG(trip_distance), 2)            AS avg_distance_miles,
        ROUND(AVG(trip_duration_minutes), 1)    AS avg_duration_min,
        COUNT(DISTINCT pu_location_id)          AS pickup_zones
    FROM {silver_table}
""").show()

StatementMeta(, 78fd09ae-500a-4d2c-bdeb-21e01f770f2a, 11, Finished, Available, Finished, False)

+-------------+-----------+-----------+------------+------------------+----------------+------------+
|earliest_date|latest_date|total_trips|avg_fare_usd|avg_distance_miles|avg_duration_min|pickup_zones|
+-------------+-----------+-----------+------------+------------------+----------------+------------+
|   2024-01-01| 2024-12-31|   35490943|       19.73|              3.56|            17.0|         262|
+-------------+-----------+-----------+------------+------------------+----------------+------------+

